# Trabajo Practico Big Data

### Importamos librerias

In [0]:
%pip uninstall -y evidently
%pip install optuna numpy==1.26.4 shap evidently==0.4.16 mpld3


In [0]:
%restart_python

In [0]:
########################
# importamos librerias #
########################

# PySpark
from pyspark.sql import functions as F
from pyspark.sql.functions import col, countDistinct
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array


# MLflow
import mlflow
from mlflow.models.signature import ModelSignature, infer_signature
from mlflow.types.schema import Schema, TensorSpec, ColSpec

# General
import numpy as np
import os
import shutil
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import mpld3

# Optuna
import optuna
import optuna.visualization

# Evidently
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, ClassificationPreset
from evidently import ColumnMapping

# Shap
import shap

### Configuración del entorno

In [0]:
# CONFIGURACIÓN — modificar según el entorno

# Dataset (Unity Catalog)
CATALOG         = "forecast"
SCHEMA          = "default"
TABLE           = "data"

# MLflow
MLFLOW_USER     = "martoschelp@gmail.com" #cambiar usuario databricks
MLFLOW_EXP_NAME = "RF_Fraud_Optuna_Spark"
UC_VOLUME_TMP   = "/Volumes/forecast/default/prueba"

#seteo directorio para archivos temportales de mlflow
os.environ["MLFLOW_DFS_TMP"] = UC_VOLUME_TMP

# Model Registry (UC)
MODEL_NAME      = "workspace.default.rf_fraud_detection"

# Optuna storage
DB_DBFS         = "/dbfs/FileStore/optuna_rf_study.db"
DB_LOCAL        = "/local_disk0/tmp/optuna_rf_study.db"

#configuro session timeout de databricks
spark.conf.set("spark.databricks.execution.timeout", 3600)


### Importamos el dataset

In [0]:
#cargamos el dataset de fraudes bancarios

df = spark.table(f'{CATALOG}.{SCHEMA}.{TABLE}')



###EDA

In [0]:
df.describe().show()
df.printSchema()
df.select(countDistinct("type")).show()
df.show(10)

In [0]:
# Tomamos una muestra para visualización (el 10% es suficiente para ver la tendencia)
pdf_eda = df.select("type", "amount", "isFraud").sample(0.1, seed=42).toPandas()

# Configuración estética
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 6))

# Hallazgo 1: Concentración de fraude por tipo
plt.subplot(1, 2, 1)
sns.countplot(data=pdf_eda, x='type', hue='isFraud')
plt.title("Frecuencia de Transacciones por Tipo")
plt.yscale('log') # Escala logarítmica para ver los fraudes (que son pocos)
plt.xticks(rotation=45)


# Hallazgo 2: Relación de montos y fraude
plt.subplot(1, 2, 2)
sns.boxplot(data=pdf_eda, x='isFraud', y='amount')
plt.title("Distribución de Montos por Clase")
plt.ylim(0, 1000000) # Limitamos para ver mejor la caja

plt.tight_layout()
mpld3.save_html(plt.gcf(), "eda_tipos_fraude.html")
plt.show()

In [0]:
# Desbalanceo de clases
fraud_counts = df.groupBy("isFraud").count().toPandas()
fraud_counts["pct"] = fraud_counts["count"] / fraud_counts["count"].sum() * 100

# ordenar por isFraud para que el orden coincida con las barras
fraud_counts = fraud_counts.sort_values("isFraud").reset_index(drop=True)

plt.figure(figsize=(6, 4))
ax = sns.barplot(data=fraud_counts, x="isFraud", y="count", order=fraud_counts["isFraud"])
plt.title("Desbalanceo de Clases")
plt.xticks([0, 1], ["Normal (0)", "Fraude (1)"])

# iterar por posición (i) que ahora sí coincide con la barra
for i, row in fraud_counts.iterrows():
    ax.text(i, row["count"] + 1000, f"{row['pct']:.2f}%", ha="center", fontsize=11)

plt.tight_layout()
mpld3.save_html(plt.gcf(), "eda_desbalanceo.html")
plt.show()

print(fraud_counts)

### Procesamiento del dataset

In [0]:
# 1. Preparación de datos

df = df.withColumn("isFraud", col("isFraud").cast("double"))

######################
# Feature engineering:
#######################

# consistencia de saldos:
df = df.withColumn("errorBalanceOrig", 
    F.col("newbalanceOrig") - F.col("oldbalanceOrg") + F.col("amount"))

df = df.withColumn("errorBalanceDest",
    F.col("newbalanceDest") - F.col("oldbalanceDest") - F.col("amount"))

# cuentas con saldo 0
df = df.withColumn("origZeroBalance",
    ((F.col("oldbalanceOrg") == 0) & (F.col("newbalanceOrig") == 0)).cast("double"))

df = df.withColumn("destZeroBalance", 
    ((F.col("oldbalanceDest") == 0) & (F.col("newbalanceDest") == 0)).cast("double"))

# Indexamos las categorías string
indexer = StringIndexer(
    inputCol="type",
    outputCol="type_idx",
    handleInvalid="keep"
)

numeric_features = [
    'step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud',
    # nuevas features
    'errorBalanceOrig', 'errorBalanceDest',
    'origZeroBalance', 'destZeroBalance'
]

feature_cols = numeric_features + ["type_idx"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df = indexer.fit(df).transform(df)

df_model = assembler.transform(df).select("features", "isFraud")
df_model = df_model.withColumnRenamed("isFraud", "label")

train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42077651)

##################
# 2. Evaluadores
##################

# curva Presisión-Recall
auc_eval = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

#metric F2 a mano porque no está en pyspark
def compute_f2_fraud(preds, fraud_label=1.0):
    """
    Calcula F2 solo para la clase fraude (label=1),
    ignorando la clase mayoritaria que infla las métricas weighted.
    """
    total_fraud   = preds.filter(F.col("label") == fraud_label).count()
    predicted_pos = preds.filter(F.col("prediction") == fraud_label).count()
    true_pos      = preds.filter(
        (F.col("label") == fraud_label) & (F.col("prediction") == fraud_label)
    ).count()

    precision = true_pos / predicted_pos if predicted_pos > 0 else 0.0
    recall    = true_pos / total_fraud   if total_fraud   > 0 else 0.0

    denom = (4 * precision + recall)
    f2    = 5 * precision * recall / denom if denom > 0 else 0.0

    return f2, precision, recall  # devolvemos los 3 para loggear en MLflow

### Optimizacion de Hiperparametros con Optuna y Registro con MLflow
Seguir en 'Levantar Modelo' si ya corriste Optuna anteriormente.

In [0]:


mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(f"/Users/{MLFLOW_USER}/{MLFLOW_EXP_NAME}")

#tomo muestra para el signature de mlflow
n_features = train_df.select("features").first()[0].size

signature = ModelSignature(
    inputs=Schema([TensorSpec(np.dtype("double"), (-1, n_features), "features")]),
    outputs=Schema([
        ColSpec("double", "prediction")
    ])
)

# 4. Función objetivo Optuna (multiobjetivo)

def objective(trial):
    params = {
        "numTrees": trial.suggest_int("numTrees", 50, 200, step=50),
        "maxDepth": trial.suggest_int("maxDepth", 3, 15),
        "maxBins": trial.suggest_int("maxBins", 32, 64, step=8),
        "minInstancesPerNode": trial.suggest_int("minInstancesPerNode", 5, 50),
        "featureSubsetStrategy": trial.suggest_categorical(
            "featureSubsetStrategy", ["sqrt", "log2", "onethird"])
    }

    rf = RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        seed=42077651,
        **params
    )

    model = rf.fit(train_df)
    preds = model.transform(test_df)

    auc            = auc_eval.evaluate(preds)
    f2, prec, rec  = compute_f2_fraud(preds, fraud_label=1.0)  # 👈 clase fraude

    with mlflow.start_run(run_name=f"optuna_trial_{trial.number}", nested=True):
        mlflow.log_params(params)
        mlflow.log_metric("pr_auc",   auc)
        mlflow.log_metric("f2_fraud",  f2)
        mlflow.log_metric("precision_fraud", prec)
        mlflow.log_metric("recall_fraud",    rec)
        mlflow.spark.log_model(model, artifact_path="RandomForest",signature=signature)

    return auc, f2

# 5. Optimización multiobjetivo

#semilla para HT con optuna
sampler = optuna.samplers.TPESampler(seed=42077651)

study = optuna.create_study(
    directions=["maximize", "maximize"],
    sampler=sampler  # ← acá va la semilla
)
#optimizamos 20 trials o 120 minutos, lo que pase primero
study.optimize(
    objective,
    n_trials=20,
    timeout=7200  # 120 minutos en segundos, dejás margen para correr el resto del codigo
)

# 6. Ver mejores trials (Pareto front)

# Con multiobjetivo no hay un único "mejor" trial,
# sino una frontera de Pareto
pareto_trials = study.best_trials  

print(f"Trials en la frontera de Pareto: {len(pareto_trials)}\n")

for trial in pareto_trials:
    print(f"Trial #{trial.number}")
    print("Params:", trial.params)
    print("AUC:", trial.values[0])
    print("F2: ", trial.values[1])
    print("-" * 50)

# Para elegir UN solo trial,
# rankear por promedio o por la métrica prioritaria
best_by_mean = max(pareto_trials, key=lambda t: (t.values[0] + t.values[1]) / 2)
best_by_f2   = max(pareto_trials, key=lambda t: t.values[1])
best_by_auc  = max(pareto_trials, key=lambda t: t.values[0])

print("\n>> Mejor por promedio AUC+F2:")
print(f"   Trial #{best_by_mean.number} | AUC={best_by_mean.values[0]:.4f} | F2={best_by_mean.values[1]:.4f}")

print("\n>> Mejor priorizando F2:")
print(f"   Trial #{best_by_f2.number}   | AUC={best_by_f2.values[0]:.4f} | F2={best_by_f2.values[1]:.4f}")

print("\n>> Mejor priorizando AUC:")
print(f"   Trial #{best_by_auc.number}  | AUC={best_by_auc.values[0]:.4f} | F2={best_by_auc.values[1]:.4f}")

###Graficos HP Tuning con Optuna

In [0]:
# Targets con nombres semánticos de tu estudio
target_prauc  = lambda t: t.values[0]  # pr_auc
target_f2     = lambda t: t.values[1]  # f2_fraud

fig1 = optuna.visualization.plot_optimization_history(
    study, target=target_prauc, target_name="PR-AUC"
)
fig1.write_html("optuna_history_prauc.html")

fig2 = optuna.visualization.plot_optimization_history(
    study, target=target_f2, target_name="F2 Fraud"
)
fig2.write_html("optuna_history_f2.html")

fig3 = optuna.visualization.plot_param_importances(
    study, target=target_prauc, target_name="PR-AUC"
)
fig3.write_html("optuna_importance_prauc.html")

fig4 = optuna.visualization.plot_param_importances(
    study, target=target_f2, target_name="F2 Fraud"
)
fig4.write_html("optuna_importance_f2.html")

fig5 = optuna.visualization.plot_parallel_coordinate(
    study, params=["numTrees", "maxDepth", "maxBins", "minInstancesPerNode"],
    target=target_prauc, target_name="PR-AUC"
)
fig5.write_html("optuna_parallel_prauc.html")

fig6 = optuna.visualization.plot_parallel_coordinate(
    study, params=["numTrees", "maxDepth", "maxBins", "minInstancesPerNode"],
    target=target_f2, target_name="F2 Fraud"
)
fig6.write_html("optuna_parallel_f2.html")

fig7 = optuna.visualization.plot_slice(
    study, params=["numTrees", "maxDepth", "maxBins", "minInstancesPerNode"],
    target=target_prauc, target_name="PR-AUC"
)
fig7.write_html("optuna_slice_prauc.html")

fig8 = optuna.visualization.plot_slice(
    study, params=["numTrees", "maxDepth", "maxBins", "minInstancesPerNode"],
    target=target_f2, target_name="F2 Fraud"
)
fig8.write_html("optuna_slice_f2.html")

fig9 = optuna.visualization.plot_pareto_front(
    study, target_names=["PR-AUC", "F2 Fraud"]
)
fig9.write_html("optuna_pareto.html")

print("✅ Todos los gráficos de Optuna guardados")

In [0]:
#graficos para optuna con multiobjetivo

# Targets con nombres semánticos de tu estudio
target_prauc  = lambda t: t.values[0]  # pr_auc
target_f2     = lambda t: t.values[1]  # f2_fraud

# --- Optimization History ---
optuna.visualization.plot_optimization_history(
    study,
    target=target_prauc,
    target_name="PR-AUC"
)

optuna.visualization.plot_optimization_history(
    study,
    target=target_f2,
    target_name="F2 Fraud"
)


In [0]:
# --- Parameter Importances ---
optuna.visualization.plot_param_importances(
    study,
    target=target_prauc,
    target_name="PR-AUC"
)

optuna.visualization.plot_param_importances(
    study,
    target=target_f2,
    target_name="F2 Fraud"
)

In [0]:
# --- Parallel Coordinate ---
# Útil para ver cómo se combinan los params en cada trial
optuna.visualization.plot_parallel_coordinate(
    study,
    params=["numTrees", "maxDepth", "maxBins", "minInstancesPerNode"],  # 👈 tus 4 params
    target=target_prauc,
    target_name="PR-AUC"
)

optuna.visualization.plot_parallel_coordinate(
    study,
    params=["numTrees", "maxDepth", "maxBins", "minInstancesPerNode"],
    target=target_f2,
    target_name="F2 Fraud"
)


In [0]:
# --- Slice Plot ---
optuna.visualization.plot_slice(
    study,
    params=["numTrees", "maxDepth", "maxBins", "minInstancesPerNode"],  
    target=target_prauc,
    target_name="PR-AUC"
)

optuna.visualization.plot_slice(
    study,
    params=["numTrees", "maxDepth", "maxBins", "minInstancesPerNode"],
    target=target_f2,
    target_name="F2 Fraud"
)


In [0]:
# --- Pareto Front (exclusivo de multi-objetivo) ---
optuna.visualization.plot_pareto_front(
    study,
    target_names=["PR-AUC", "F2 Fraud"]  
)

###Registro del mejor modelo
Reentreno y registro en ML

In [0]:
best_trial = best_by_mean

with mlflow.start_run(run_name="RandomForest_Final") as run:
    best_params = best_trial.params

    rf_final = RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        seed=42077651,
        **best_params
    )

    model_final = rf_final.fit(train_df)
    preds_final = model_final.transform(test_df)

    auc_final = auc_eval.evaluate(preds_final)
    f2_final, prec_final, rec_final = compute_f2_fraud(preds_final, fraud_label=1.0)

    mlflow.log_params(best_params)
    mlflow.log_metric("pr_auc",          auc_final)
    mlflow.log_metric("f2_fraud",        f2_final)
    mlflow.log_metric("precision_fraud", prec_final)
    mlflow.log_metric("recall_fraud",    rec_final)
    mlflow.log_param("selection_criteria",  "best_mean_pareto")
    mlflow.log_param("optuna_trial_number", best_trial.number)

    mlflow.spark.log_model(
        model_final,
        artifact_path="RandomForest",
        signature=signature
    )

    mlflow.register_model(
        model_uri=f"runs:/{run.info.run_id}/RandomForest",  
        name=MODEL_NAME
    )

    print(f"Modelo final registrado | AUC={auc_final:.4f} | F2={f2_final:.4f}")

##Levantar modelo registrado
Seguir por aca si ya corriste HP Tuning y registraste el modelo

In [0]:
# ─────────────────────────────────────────────
# LOAD MODEL FROM REGISTRY (skip HP tuning)
# ─────────────────────────────────────────────
import mlflow

# 1. validar dataset
if "test_df" not in globals():
    raise RuntimeError("❌ test_df no está definido — corré las celdas previas")

mlflow.set_registry_uri("databricks-uc")

# Load from Unity Catalog using fully qualified name and version
# dfs_tmpdir parameter is required for SparkML models on serverless compute
loaded_model = mlflow.spark.load_model(f"models:/{MODEL_NAME}/3", dfs_tmpdir=UC_VOLUME_TMP)

# Regenerar predicciones sobre test_df (ya tiene que estar construido)
preds_final = loaded_model.transform(test_df)

# Verificar que funciona
auc_check = auc_eval.evaluate(preds_final)
f2_check, prec_check, rec_check = compute_f2_fraud(preds_final, fraud_label=1.0)

print(f"Modelo cargado OK | AUC={auc_check:.4f} | F2={f2_check:.4f}")

In [0]:
print("loaded_model" in globals())

###Explicacion de uso: Inferencia con el modelo registrado

In [0]:
# 1. Definir la ruta del modelo en el Registry
# Usamos la variable MODEL_NAME que ya definiste antes

model_uri = f"models:/{MODEL_NAME}/{3}"

print(f"Cargando el modelo desde: {model_uri}")
loaded_model = mlflow.spark.load_model(model_uri)

# 2. Crear una "Nueva Transacción" de ejemplo
# En la vida real, esto vendría de una API o un streaming. 
# Aquí tomamos una fila del set de test y le quitamos la etiqueta real para simular que es nueva.
new_transaction = test_df.limit(1).drop("isFraud", "label") 
print('new_transaction: no fraude')
new_transaction.show(truncate=False)

new_transaction_fraud = test_df.filter(F.col("label") == 1.0).limit(1).drop("label")
print('new_transaction_fraude: fraude')
new_transaction_fraud.show(truncate=False)

# 3. Ejecutar la predicción
prediction_result = loaded_model.transform(new_transaction)

# 4. Mostrar el resultado de forma legible
# Cambiamos la selección para usar solo lo que el modelo devolvió seguro
res = prediction_result.select("probability", "prediction").collect()[0]

print("-" * 50)
print(f"RESULTADO DEL MODELO:")
status = "⚠️ FRAUDE DETECTADO" if res['prediction'] == 1.0 else "✅ TRANSACCIÓN NORMAL"
prob = res['probability'][1] if res['prediction'] == 1.0 else res['probability'][0]

print(f"Veredicto: {status}")
print(f"Confianza: {prob*100:.2f}%")

print("-" * 50)

###Fraude
# 3. Ejecutar la predicción
prediction_result = loaded_model.transform(new_transaction_fraud)

# 4. Mostrar el resultado de forma legible
# Cambiamos la selección para usar solo lo que el modelo devolvió seguro
res = prediction_result.select("probability", "prediction").collect()[0]

print("-" * 50)
print(f"RESULTADO DEL MODELO:")
status = "⚠️ FRAUDE DETECTADO" if res['prediction'] == 1.0 else "✅ TRANSACCIÓN NORMAL"
prob = res['probability'][1] if res['prediction'] == 1.0 else res['probability'][0]

print(f"Veredicto: {status}")
print(f"Confianza: {prob*100:.2f}%")

print("-" * 50)

###SHAP

In [0]:
# 1. Definir las 12 columnas exactas
columnas_features = [
    'step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud',
    'errorBalanceOrig', 'errorBalanceDest',
    'origZeroBalance', 'destZeroBalance', 'type_idx'
]

# 2. Tomar muestra de test_df
# Importante: Como test_df ya tiene la columna 'features', la usamos directamente
sample_data = test_df.select("features").limit(20).toPandas()
X_sample = np.array(sample_data['features'].tolist())

# 3. Función puente CORREGIDA
# Ahora incluimos el ensamblador para que el modelo no se queje de la falta de 'features'
def predict_proba_spark(data_numpy):
    # Convertimos numpy a pandas
    pdf = pd.DataFrame(data_numpy, columns=columnas_features)
    sdf = spark.createDataFrame(pdf)
    
    # ENSAMBLAMOS las columnas en un vector llamado 'features'
    # Esto es lo que le faltaba al modelo
    assembler_shap = VectorAssembler(inputCols=columnas_features, outputCol="features")
    sdf_with_features = assembler_shap.transform(sdf)
    
    # Ahora sí, el modelo encontrará su columna 'features'
    preds = loaded_model.transform(sdf_with_features)
    
    # Extraemos la probabilidad de fraude
    res = preds.select("probability").collect()
    return np.array([float(row.probability[1]) for row in res])

# 4. Configurar el fondo
background = shap.kmeans(X_sample, 5)

# 5. Inicializar Explainer
explainer = shap.KernelExplainer(predict_proba_spark, background, link="identity")

# 6. Calcular valores SHAP (esto toma 1-2 minutos)
shap_values = explainer.shap_values(X_sample)

# 7. Graficar
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_sample, feature_names=columnas_features, show=False)
plt.title("Importancia de Variables (SHAP) - Explicación del Modelo")
mpld3.save_html(plt.gcf(), "SHAP.html")
plt.show()

In [0]:
# ================================
# 2. PREDICCIONES
# ================================
# REQUISITO: loaded_model y test_df ya definidos
preds_final = loaded_model.transform(test_df)

In [0]:
# ================================
# 3. CONVERSION SEGURA A PANDAS
# ================================

MAX_ROWS = 50000

# ================================
# 3.1. expandir features en Spark
# ================================
train_df_exp = train_df.withColumn("features_array", vector_to_array("features"))
test_df_exp  = test_df.withColumn("features_array", vector_to_array("features"))

# ================================
# 3.2. convertir a pandas (DESDE expandidos)
# ================================
train_pd = train_df_exp.limit(MAX_ROWS).toPandas()
test_pd  = test_df_exp.limit(MAX_ROWS).toPandas()
preds_pd = preds_final.limit(MAX_ROWS).toPandas()

# ================================
# 3.3. expandir array en columnas reales
# ================================
train_features = pd.DataFrame(train_pd["features_array"].tolist())
test_features  = pd.DataFrame(test_pd["features_array"].tolist())

train_pd = pd.concat([train_pd.drop(columns=["features", "features_array"]), train_features], axis=1)
test_pd  = pd.concat([test_pd.drop(columns=["features", "features_array"]), test_features], axis=1)

In [0]:
# ⚠️ Evita romper por memoria

#train_pd = train_df.limit(MAX_ROWS).toPandas()
#test_pd  = test_df.limit(MAX_ROWS).toPandas()
#preds_pd = preds_final.limit(MAX_ROWS).toPandas()

In [0]:
# ================================
# 4. LIMPIEZA DE COLUMNAS SPARK
# ================================
for df in [train_pd, test_pd, preds_pd]:
    for col in ["features", "rawPrediction", "probability"]:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)

# ================================
# 5. ASEGURAR COLUMNAS NUMERICAS
# ================================
train_pd = train_pd.select_dtypes(include=["number"])
test_pd  = test_pd.select_dtypes(include=["number"])

# ================================
# 6. FORZAR TIPOS NUMERICOS
# ================================
train_pd = train_pd.apply(pd.to_numeric, errors="coerce")
test_pd  = test_pd.apply(pd.to_numeric, errors="coerce")

# ================================
# 7. ELIMINAR COLUMNAS ROTAS
# ================================
train_pd = train_pd.dropna(axis=1, how="all")
test_pd  = test_pd.dropna(axis=1, how="all")

# ================================
# 8. ALINEAR COLUMNAS (CLAVE)
# ================================
common_cols = train_pd.columns.intersection(test_pd.columns)

train_pd = train_pd[common_cols]
test_pd  = test_pd[common_cols]

# ================================
# 9. 🔥 FIX FINAL (CLAVE)
# ================================
# evitar error: '<' between str and int
train_pd.columns = train_pd.columns.astype(str)
test_pd.columns  = test_pd.columns.astype(str)

In [0]:
# ================================
# 6. DATA DRIFT (TRAIN vs TEST)
# ================================
drift_report = Report(metrics=[DataDriftPreset()])

drift_report.run(
    reference_data=train_pd,
    current_data=test_pd
)

# guardar en local (NO DBFS)
drift_path = "drift_report.html"
drift_report.save_html(drift_path)

# mostrar
displayHTML(open(drift_path).read())

In [0]:
print(test_pd.dtypes)

In [0]:
# ================================
# 7. DETECCION DINAMICA DE TARGET
# ================================
# ⚠️ Esto evita que se rompa si no es "label"

print("Columnas en preds:", preds_pd.columns)

possible_targets = ["label", "target", "y", "is_fraud"]

target_col = None
for col in possible_targets:
    if col in preds_pd.columns:
        target_col = col
        break

if target_col is None:
    raise ValueError("❌ No se encontró columna target en preds_pd")

print(f"✅ Usando columna target: {target_col}")

In [0]:
# ================================
# 8. PREPARAR DATA DE PERFORMANCE
# ================================
eval_df = preds_pd[["prediction", target_col]].dropna().copy()

# asegurar tipos correctos
eval_df["prediction"] = eval_df["prediction"].astype(float).astype(int)
eval_df[target_col]   = eval_df[target_col].astype(float).astype(int)

# rename requerido por evidently
eval_df.rename(columns={target_col: "target"}, inplace=True)

In [0]:
# ================================
# 9. PERFORMANCE DEL MODELO (SIN COMPARACION REAL)
# ================================


# -------------------------------
# VALIDACION
# -------------------------------
if not {"prediction", "label"}.issubset(preds_pd.columns):
    raise ValueError(f"❌ preds_pd debe tener ['prediction', 'label']")

# -------------------------------
# ARMAR DATASET DE EVALUACION
# -------------------------------
eval_df = preds_pd[["prediction", "label"]].dropna().copy()

# asegurar tipos
eval_df["prediction"] = pd.to_numeric(eval_df["prediction"], errors="coerce").astype(float).astype(int)
eval_df["label"]      = pd.to_numeric(eval_df["label"], errors="coerce").astype(float).astype(int)

# rename para evidently
eval_df.rename(columns={"label": "target"}, inplace=True)

# -------------------------------
# COLUMN MAPPING (CLAVE)
# -------------------------------
column_mapping = ColumnMapping()
column_mapping.target = "target"
column_mapping.prediction = "prediction"

# -------------------------------
# EVIDENTLY REPORT
# -------------------------------
perf_report = Report(metrics=[ClassificationPreset()])

perf_report.run(
    reference_data=eval_df,
    current_data=eval_df,  # mismo dataset → solo métricas
    column_mapping=column_mapping
)

# -------------------------------
# GUARDAR Y MOSTRAR (SIN DBFS)
# -------------------------------
perf_path = "performance_report.html"
perf_report.save_html(perf_path)

displayHTML(open(perf_path).read())

In [0]:
# ================================
# SIMULACION DE DRIFT (BALANCEADA)
# ================================


test_drifted = test_pd.copy()

cols = [col for col in test_drifted.columns if col != "label"]

# -------------------------------
# 1. drift leve (shift chico)
# -------------------------------
for col in cols[:3]:
    test_drifted[col] = test_drifted[col] * 1.1

# -------------------------------
# 2. drift moderado
# -------------------------------
for col in cols[3:6]:
    test_drifted[col] = test_drifted[col] + test_drifted[col].mean() * 0.2

# -------------------------------
# 3. ruido leve
# -------------------------------
for col in cols[6:9]:
    noise = np.random.normal(0, test_drifted[col].std() * 0.2, size=len(test_drifted))
    test_drifted[col] += noise


# -------------------------------
# correr drift (test vs drifted)
# -------------------------------
drift_sim_report = Report(metrics=[DataDriftPreset()])

drift_sim_report.run(
    reference_data=test_pd,
    current_data=test_drifted
)

# guardar local
sim_path = "drift_simulation.html"
drift_sim_report.save_html(sim_path)

displayHTML(open(sim_path).read())

###Serving del Modelo para aser invocado por fuera de Databricks
Se registra el modelo con pandas y sklearn por errores con pyspark y java

In [0]:
from mlflow import MlflowClient

client = MlflowClient(registry_uri="databricks-uc")

# traer los params del run donde registraste el modelo
model_version = client.get_model_version(MODEL_NAME, "3")
run_id = model_version.run_id

run = client.get_run(run_id)
best_params = run.data.params
print(best_params)

In [0]:
from sklearn.ensemble import RandomForestClassifier as SklearnRF
from sklearn.metrics import average_precision_score, fbeta_score, precision_score, recall_score
import mlflow.sklearn

# ================================
# 1. CONVERTIR A PANDAS
# (si ya lo tenés de antes, salteá este bloque)
# ================================
MAX_ROWS = 50000

feature_cols = [
    'step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud',
    'errorBalanceOrig', 'errorBalanceDest',
    'origZeroBalance', 'destZeroBalance', 'type_idx'
]

from pyspark.ml.functions import vector_to_array

train_pd_sk = (
    train_df
    .withColumn("features_array", vector_to_array("features"))
    .limit(MAX_ROWS)
    .toPandas()
)
test_pd_sk = (
    test_df
    .withColumn("features_array", vector_to_array("features"))
    .limit(MAX_ROWS)
    .toPandas()
)

# expandir vector en columnas
import pandas as pd
import numpy as np

train_features_sk = pd.DataFrame(train_pd_sk["features_array"].tolist(), columns=feature_cols)
test_features_sk  = pd.DataFrame(test_pd_sk["features_array"].tolist(),  columns=feature_cols)

X_train = train_features_sk
y_train = train_pd_sk["label"].astype(int)

X_test  = test_features_sk
y_test  = test_pd_sk["label"].astype(int)

# ================================
# 2. REENTRENAR CON SKLEARN
# usando los mejores params de Optuna
# ================================

# mapeo de valores PySpark -> sklearn
feature_strategy_map = {
    "sqrt"     : "sqrt",
    "log2"     : "log2",
    "onethird" : 0.333,  # equivalente manual
    "auto"     : "sqrt",
    "all"      : 1.0
}

best_params_typed = {
    "numTrees"              : int(best_params["numTrees"]),
    "maxDepth"              : int(best_params["maxDepth"]),
    "maxBins"               : int(best_params["maxBins"]),
    "minInstancesPerNode"   : int(best_params["minInstancesPerNode"]),
    "featureSubsetStrategy" : feature_strategy_map.get(best_params["featureSubsetStrategy"], "sqrt")
}

print(best_params_typed)

rf_sklearn = SklearnRF(
    n_estimators     = best_params_typed["numTrees"],
    max_depth        = best_params_typed["maxDepth"],
    max_features     = best_params_typed["featureSubsetStrategy"],
    min_samples_leaf = best_params_typed["minInstancesPerNode"],
    n_jobs           = -1,
    random_state     = 42077651
)


rf_sklearn.fit(X_train, y_train)

# ================================
# 3. EVALUAR
# ================================
y_pred      = rf_sklearn.predict(X_test)
y_proba     = rf_sklearn.predict_proba(X_test)[:, 1]

pr_auc    = average_precision_score(y_test, y_proba)
f2        = fbeta_score(y_test, y_pred, beta=2, pos_label=1)
precision = precision_score(y_test, y_pred, pos_label=1)
recall    = recall_score(y_test, y_pred, pos_label=1)

print(f"PR-AUC:    {pr_auc:.4f}")
print(f"F2 fraud:  {f2:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")

# ================================
# 4. REGISTRAR EN MLFLOW (sklearn)
# ================================
MODEL_NAME_SK = "workspace.default.rf_fraud_sklearn"

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(f"/Users/{MLFLOW_USER}/{MLFLOW_EXP_NAME}")

with mlflow.start_run(run_name="RF_sklearn_serving") as run:
    mlflow.log_params(best_params)
    mlflow.log_metric("pr_auc",          pr_auc)
    mlflow.log_metric("f2_fraud",        f2)
    mlflow.log_metric("precision_fraud", precision)
    mlflow.log_metric("recall_fraud",    recall)
    mlflow.log_param("selection_criteria", "best_mean_pareto")

    mlflow.sklearn.log_model(
        rf_sklearn,
        artifact_path="RandomForest_sklearn",
        input_example=X_test.iloc[:3],
        registered_model_name=MODEL_NAME_SK
    )

    print(f"Modelo sklearn registrado: {MODEL_NAME_SK}")
    print(f"Run ID: {run.info.run_id}")

# ================================
# 5. ASIGNAR ALIAS champion
# ================================
from mlflow import MlflowClient

client = MlflowClient(registry_uri="databricks-uc")
client.set_registered_model_alias(
    name=MODEL_NAME_SK,
    alias="champion",
    version="1"
)
print("Alias @champion asignado")